# `find_plane_normal()`

`nematics3d.find_plane_normal()` fits a least-squares plane to a finite 3D point cloud and returns its unit normal together with diagnostics for judging the fit.

## What `find_plane_normal()` is for

Use this function when a collection of 3D points is expected to lie approximately in one plane and you need a representative plane orientation. The fitted plane passes through the point-cloud centroid, and its normal is chosen to minimize the summed squared perpendicular distances from the points to that plane.

The returned normal is an **axis**, not an oriented vector: `normal` and `-normal` describe the same plane. The result also reports the cloud thickness and degeneracy diagnostics, which are important when the points are noisy or nearly collinear.

## Setup

**This section can be safely skipped.** Run the following cell to import NumPy and Nematics3D.

In [ ]:
import numpy as np
import nematics3d as n3d

## Minimal example

The points below lie exactly on the plane $z=0.5x-0.25y+2$. Only the point coordinates are required.

In [ ]:
x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0, 0.5])
y = np.array([1.0, -2.0, 0.5, 2.0, -1.0, -1.5])
z = 0.5 * x - 0.25 * y + 2.0
points = np.column_stack([x, y, z])

result = n3d.find_plane_normal(points)
result

The returned normal is perpendicular to the fitted plane. For this exact planar cloud, `planarity_score` is 1 and `thickness_rms` is 0 up to floating-point roundoff.

### What is returned

`find_plane_normal()` returns one `PlaneNormalResult`. Its main field is `result.normal`, the fitted unit normal. The same object also contains the centroid and diagnostics describing the geometry of the fit.

### Reading the output

Do not use the sign of `result.normal` to distinguish two plane orientations. If two fitted normals should represent the same plane axis, compare them with `abs(np.dot(n1, n2))`; a value near 1 means the axes agree.

For a well-defined planar cloud, look for a small `thickness_rms`, a `planarity_score` near 1, and a `linearity_risk` well below 1. A high planarity score alone is not enough when the cloud is nearly a line.

## What is the next step?

The fitted normal can be used anywhere a representative plane orientation is needed—for example, to construct a local coordinate frame, compare orientations, or project a nearly planar point cloud. If the points are noisy or elongated, inspect the returned diagnostics before treating the normal as geometrically well determined.

## Arguments

```python
find_plane_normal(points)
```

| Argument | What it controls | Typical form |
| --- | --- | --- |
| `points` | The 3D point cloud used to fit the plane. At least three finite points are required. | Array-like with shape `(N, 3)` |

`points` may contain integer or floating-point coordinates. Inputs with fewer than three points, non-finite values, or a coordinate dimension other than 3 are rejected.

The function does not require the input to define a unique plane. Collinear or coincident points are accepted, but the returned normal is then geometrically underdetermined; use the diagnostics described below to identify that situation.

## Special examples

### Example: a noisy plane

Real point clouds usually have finite thickness. Here Gaussian noise is added to an otherwise planar cloud.

In [ ]:
rng = np.random.default_rng(7)
x = rng.uniform(-2.0, 2.0, 200)
y = rng.uniform(-2.0, 2.0, 200)
z = 0.5 * x - 0.25 * y + 2.0 + rng.normal(scale=0.05, size=200)
noisy_points = np.column_stack([x, y, z])

noisy_result = n3d.find_plane_normal(noisy_points)
print("normal:", noisy_result.normal)
print("planarity_score:", noisy_result.planarity_score)
print("thickness_rms:", noisy_result.thickness_rms)
print("linearity_risk:", noisy_result.linearity_risk)

`thickness_rms` is now nonzero and has the same length units as the input coordinates. The normal can still be well determined when the cloud is not exactly planar.

### Example: a nearly one-dimensional cloud

A straight line lies in infinitely many planes. This is an important degeneracy: such a cloud can have `planarity_score == 1` even though there is no unique plane normal.

In [ ]:
t = np.linspace(-2.0, 2.0, 8)
line_points = np.column_stack([t, np.zeros_like(t), np.zeros_like(t)])
line_result = n3d.find_plane_normal(line_points)

print("normal:", line_result.normal)
print("planarity_score:", line_result.planarity_score)
print("linearity_risk:", line_result.linearity_risk)
print("eigenvalues:", line_result.eigenvalues)

For an exactly one-dimensional cloud, `linearity_risk` is 1. Treat the reported normal as underdetermined even though `planarity_score` is also 1.

## Returned objects

### `PlaneNormalResult`

The result is a structured `ResultBase` object with the following fields:

| Field | Meaning |
| --- | --- |
| `normal` | Unit normal of the least-squares best-fit plane. Its sign is arbitrary. |
| `centroid` | Mean of the input coordinates; the fitted plane passes through this point. |
| `planarity_score` | Dimensionless score in `[0, 1]`; values near 1 mean little variance lies normal to the fitted plane. |
| `thickness_rms` | RMS thickness along the fitted normal, in the same length units as `points`. |
| `linearity_risk` | Ratio of the two smallest second-moment eigenvalues; values near 1 indicate that the normal is poorly determined because the cloud is close to one-dimensional. |
| `eigenvalues` | Three ascending eigenvalues of the centered point-cloud second-moment matrix. |

`result.metric` returns the diagnostics other than `normal` as a shallow dictionary.

## Details

Let the input points be $\mathbf{x}_i$ and let their centroid be

$$\bar{\mathbf{x}}=\frac{1}{N}\sum_i\mathbf{x}_i. $$

With centered coordinates $\mathbf{r}_i=\mathbf{x}_i-\bar{\mathbf{x}}$, the function forms the second-moment matrix

$$M=\sum_i \mathbf{r}_i\mathbf{r}_i^T. $$

For a unit vector $\mathbf{n}$,

$$\mathbf{n}^T M \mathbf{n}=\sum_i(\mathbf{r}_i\cdot\mathbf{n})^2. $$

Therefore the eigenvector belonging to the smallest eigenvalue minimizes the summed squared perpendicular distance to a plane through the centroid. This eigenvector is returned as `normal`.

For ascending eigenvalues $\lambda_0\leq\lambda_1\leq\lambda_2$, the diagnostics are

$$\mathrm{planarity\_score}=\operatorname{clip}\left(1-\frac{3\lambda_0}{\lambda_0+\lambda_1+\lambda_2},0,1\right),$$

$$\mathrm{thickness\_rms}=\sqrt{\frac{\lambda_0}{N}},$$

and, when $\lambda_1>0$,

$$\mathrm{linearity\_risk}=\frac{\lambda_0}{\lambda_1}. $$

If $\lambda_1=0$, `linearity_risk` is defined as 1. For a coincident cloud the total variance is zero; `planarity_score` is defined as 1, but the normal is not geometrically meaningful.